In [1]:
pip install pyqubo openjij numpy pandas matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 136.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 3.7 MB/s  0:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 2.0 MB/s  0:00:05
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 47.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 103.5 MB/s  0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [pyqubo]


In [3]:
import time
import itertools
import numpy as np
import pandas as pd
from pyqubo import Array, Constraint, Placeholder
import openjij as oj

# ==========================================
# 1. DEFINE RAILWAY TIMETABLE & CLEARANCE
# ==========================================
# Scenario 1: 4 Trains, 2 Platforms (8 Binary Variables)
trains_data = [
    {"id": "T0 (Express)",   "arr": 10.00, "dep": 10.20, "pref_plat": 0, "priority": 3},
    {"id": "T1 (Passenger)", "arr": 10.12, "dep": 10.28, "pref_plat": 0, "priority": 1},
    {"id": "T2 (Superfast)", "arr": 10.25, "dep": 10.45, "pref_plat": 1, "priority": 4},
    {"id": "T3 (Freight)",   "arr": 10.36, "dep": 10.55, "pref_plat": 1, "priority": 1},
]
num_trains = len(trains_data)
num_platforms = 2
clearance_time = 0.05  # Minimum clearance between trains

# Automatically detect overlapping intervals (using round() to prevent float precision bugs)
conflicts = []
for i in range(num_trains):
    for j in range(i + 1, num_trains):
        start_i, end_i = round(trains_data[i]["arr"], 4), round(trains_data[i]["dep"] + clearance_time, 4)
        start_j, end_j = round(trains_data[j]["arr"], 4), round(trains_data[j]["dep"] + clearance_time, 4)
        if max(start_i, start_j) < min(end_i, end_j):
            conflicts.append((i, j))

print(f"Auto-detected Conflict Pairs: {conflicts}")

# Generate Platform Assignment Cost Matrix
platform_cost = np.zeros((num_trains, num_platforms))
for i, t in enumerate(trains_data):
    for p in range(num_platforms):
        if p == t["pref_plat"]:
            platform_cost[i, p] = 1.0 * t["priority"]
        else:
            platform_cost[i, p] = 4.0 * t["priority"]

# ==========================================
# 2. FORMULATE SYMBOLIC QUBO WITH PYQUBO
# ==========================================
x = Array.create('x', shape=(num_trains, num_platforms), vartype='BINARY')
penalty_lambda = Placeholder('lambda')

H_cost = sum(platform_cost[i, p] * x[i, p]
             for i in range(num_trains) for p in range(num_platforms))

H_onehot = sum(Constraint((sum(x[i, p] for p in range(num_platforms)) - 1)**2,
                          label=f"onehot_T{i}")
               for i in range(num_trains))

H_conflict = sum(Constraint(x[i, p] * x[j, p],
                            label=f"conflict_T{i}_T{j}_P{p}")
                 for (i, j) in conflicts for p in range(num_platforms))

H = H_cost + penalty_lambda * (H_onehot + H_conflict)
model = H.compile()

feed_dict = {'lambda': 15.0}
qubo, offset = model.to_qubo(feed_dict=feed_dict)

# ==========================================
# 3. HELPER: EVALUATE SAMPLER RESPONSES
# ==========================================
def evaluate_response(response, model, feed_dict, runtime_ms):
    valid_costs = []
    best_sample_decoded = None
    best_cost = float('inf')
    total_occurrences = sum(response.record.num_occurrences)

    for sample_arr, energy, num_occ in response.record:
        sample_dict = {response.variables[k]: int(sample_arr[k]) for k in range(len(response.variables))}
        decoded = model.decode_sample(sample_dict, vartype='BINARY', feed_dict=feed_dict)
        broken = decoded.constraints(only_broken=True)

        if len(broken) == 0:
            op_cost = sum(platform_cost[i, p] * decoded.array('x', (i, p))
                          for i in range(num_trains) for p in range(num_platforms))
            valid_costs.extend([op_cost] * num_occ)
            if op_cost < best_cost:
                best_cost = op_cost
                best_sample_decoded = decoded

    feasibility_pct = (len(valid_costs) / total_occurrences) * 100 if total_occurrences > 0 else 0.0
    avg_valid_cost = np.mean(valid_costs) if valid_costs else float('nan')
    return {
        "Valid (%)": round(feasibility_pct, 2),
        "Best Valid Cost": best_cost if valid_costs else None,
        "Avg Valid Cost": round(avg_valid_cost, 2) if valid_costs else None,
        "Runtime (ms)": round(runtime_ms, 2),
        "Best Decoded": best_sample_decoded
    }

# ==========================================
# 4. RUN ALL 3 SOLVERS (EXACT, SA, SQA)
# ==========================================
# A. Exact Classical Enumeration
t0 = time.perf_counter()
best_exact_cost = float('inf')
valid_exact_costs = []
for bits in itertools.product([0, 1], repeat=num_trains * num_platforms):
    grid = np.array(bits).reshape((num_trains, num_platforms))
    if not np.all(grid.sum(axis=1) == 1):
        continue
    if any(grid[i, p] == 1 and grid[j, p] == 1 for (i, j) in conflicts for p in range(num_platforms)):
        continue
    cost = np.sum(grid * platform_cost)
    valid_exact_costs.append(cost)
    if cost < best_exact_cost:
        best_exact_cost = cost
exact_time_ms = (time.perf_counter() - t0) * 1000

# B. Classical Simulated Annealing (OpenJij SASampler)
sa_sampler = oj.SASampler()
t0 = time.perf_counter()
sa_res = sa_sampler.sample_qubo(qubo, num_reads=500)
sa_time_ms = (time.perf_counter() - t0) * 1000
sa_metrics = evaluate_response(sa_res, model, feed_dict, sa_time_ms)

# C. Simulated Quantum Annealing (OpenJij SQASampler)
sqa_sampler = oj.SQASampler()
t0 = time.perf_counter()
sqa_res = sqa_sampler.sample_qubo(qubo, num_reads=500)
sqa_time_ms = (time.perf_counter() - t0) * 1000
sqa_metrics = evaluate_response(sqa_res, model, feed_dict, sqa_time_ms)

# ==========================================
# 5. PRINT SUMMARY TABLE & WINNING SCHEDULE
# ==========================================
summary_df = pd.DataFrame([
    {
        "Solver": "Exact Classical Search",
        "Valid (%)": round((len(valid_exact_costs) / (2**(num_trains * num_platforms))) * 100, 2),
        "Best Valid Cost": best_exact_cost if valid_exact_costs else None,
        "Avg Valid Cost": round(np.mean(valid_exact_costs), 2) if valid_exact_costs else None,
        "Runtime (ms)": round(exact_time_ms, 2)
    },
    {"Solver": "Classical SA (OpenJij)", **{k: v for k, v in sa_metrics.items() if k != "Best Decoded"}},
    {"Solver": "Simulated QA (OpenJij)", **{k: v for k, v in sqa_metrics.items() if k != "Best Decoded"}},
])
print("\n--- SOLVER COMPARISON (SCENARIO 1: 4 TRAINS, 2 PLATFORMS) ---")
print(summary_df.to_string(index=False))

print("\n--- OPTIMAL SQA PLATFORM ASSIGNMENT ---")
best_sqa = sqa_metrics["Best Decoded"]
if best_sqa is not None:
    for i, t in enumerate(trains_data):
        for p in range(num_platforms):
            if best_sqa.array('x', (i, p)) == 1:
                status = "Kept Preferred" if p == t["pref_plat"] else "Reassigned"
                print(f"{t['id']} [{t['arr']:.2f} - {t['dep']:.2f}] -> Platform {p} ({status})")
else:
    print("No feasible schedule found! Check if the number of overlapping trains exceeds available platforms.")

Auto-detected Conflict Pairs: [(0, 1), (1, 2), (2, 3)]

--- SOLVER COMPARISON (SCENARIO 1: 4 TRAINS, 2 PLATFORMS) ---
                Solver  Valid (%)  Best Valid Cost  Avg Valid Cost  Runtime (ms)
Exact Classical Search       0.78             21.0            22.5         10.89
Classical SA (OpenJij)      69.40             21.0            21.0        410.03
Simulated QA (OpenJij)      77.20             21.0            21.0       4253.46

--- OPTIMAL SQA PLATFORM ASSIGNMENT ---
T0 (Express) [10.00 - 10.20] -> Platform 1 (Reassigned)
T1 (Passenger) [10.12 - 10.28] -> Platform 0 (Kept Preferred)
T2 (Superfast) [10.25 - 10.45] -> Platform 1 (Kept Preferred)
T3 (Freight) [10.36 - 10.55] -> Platform 0 (Reassigned)


In [5]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyqubo import Array, Constraint, Placeholder
import openjij as oj

np.random.seed(42)

# ==========================================
# CORE MODULE: GUARANTEED-SOLVABLE CONFLICT GENERATOR
# ==========================================
def generate_railway_scenario(num_trains, num_platforms, clearance=0.05, seed=42):
    """
    Generates a delayed railway timetable where trains have realistic pairwise
    overlaps, while guaranteeing at least one conflict-free platform assignment exists.
    """
    rng = np.random.default_rng(seed)
    trains = []
    # Track when each platform becomes free in a hidden ground-truth schedule
    plat_free_time = np.full(num_platforms, 10.00)

    for i in range(num_trains):
        # Pick the earliest available platform for ground-truth feasibility
        true_p = int(np.argmin(plat_free_time))
        # Stagger arrivals slightly so trains on OTHER platforms overlap with this train
        arr = round(plat_free_time[true_p] + rng.uniform(0.01, 0.06), 2)
        dwell = round(rng.uniform(0.15, 0.22), 2)
        dep = round(arr + dwell, 2)
        plat_free_time[true_p] = round(dep + clearance + 0.01, 2)

        # Because of "delays", the train's originally preferred platform is often congested
        pref_plat = int(rng.integers(0, num_platforms))
        priority = int(rng.choice([1, 2, 3, 4], p=[0.4, 0.3, 0.2, 0.1]))
        trains.append({
            "id": f"T{i}", "arr": arr, "dep": dep,
            "pref_plat": pref_plat, "priority": priority
        })

    # Sort trains by arrival time
    trains = sorted(trains, key=lambda x: x["arr"])
    for idx, t in enumerate(trains):
        t["id"] = f"T{idx}"

    # Automatically detect overlapping intervals (including clearance)
    conflicts = []
    for i in range(num_trains):
        for j in range(i + 1, num_trains):
            s_i, e_i = round(trains[i]["arr"], 4), round(trains[i]["dep"] + clearance, 4)
            s_j, e_j = round(trains[j]["arr"], 4), round(trains[j]["dep"] + clearance, 4)
            if max(s_i, s_j) < min(e_i, e_j):
                conflicts.append((i, j))

    # Build Platform Assignment Cost Matrix
    platform_cost = np.zeros((num_trains, num_platforms))
    for i, t in enumerate(trains):
        for p in range(num_platforms):
            platform_cost[i, p] = (1.0 if p == t["pref_plat"] else 4.0) * t["priority"]

    return trains, conflicts, platform_cost

def build_compiled_model(num_trains, num_platforms, conflicts, platform_cost):
    x = Array.create('x', shape=(num_trains, num_platforms), vartype='BINARY')
    lam = Placeholder('lambda')

    H_cost = sum(platform_cost[i, p] * x[i, p]
                 for i in range(num_trains) for p in range(num_platforms))
    H_onehot = sum(Constraint((sum(x[i, p] for p in range(num_platforms)) - 1)**2, label=f"oh_{i}")
                   for i in range(num_trains))
    H_conf = sum(Constraint(x[i, p] * x[j, p], label=f"cf_{i}_{j}_{p}")
                 for (i, j) in conflicts for p in range(num_platforms))

    H = H_cost + lam * (H_onehot + H_conf)
    return H.compile()

def fast_evaluate(response, num_trains, num_platforms, conflicts, platform_cost):
    var_map = {v: idx for idx, v in enumerate(response.variables)}
    idx_grid = np.zeros((num_trains, num_platforms), dtype=int)
    for i in range(num_trains):
        for p in range(num_platforms):
            idx_grid[i, p] = var_map[f"x[{i}][{p}]"]

    valid_costs = []
    best_cost = float('inf')
    best_grid = None
    total_occ = sum(response.record.num_occurrences)

    for sample_arr, energy, num_occ in response.record:
        grid = sample_arr[idx_grid]
        if not np.all(grid.sum(axis=1) == 1):
            continue
        if any(grid[i, p] == 1 and grid[j, p] == 1 for (i, j) in conflicts for p in range(num_platforms)):
            continue
        cost = float(np.sum(grid * platform_cost))
        valid_costs.extend([cost] * num_occ)
        if cost < best_cost:
            best_cost = cost
            best_grid = grid.copy()

    valid_pct = (len(valid_costs) / total_occ) * 100.0 if total_occ > 0 else 0.0
    avg_cost = float(np.mean(valid_costs)) if valid_costs else np.nan
    return valid_pct, (best_cost if valid_costs else np.nan), avg_cost, best_grid

# ==========================================
# EXPERIMENT 1: BEFORE vs AFTER GANTT CHART
# ==========================================
print("Running Experiment 1: Before vs. After Platform Allocation...")
trains_1 = [
    {"id": "T0 (Express)",   "arr": 10.00, "dep": 10.20, "pref_plat": 0, "priority": 3},
    {"id": "T1 (Passenger)", "arr": 10.12, "dep": 10.28, "pref_plat": 0, "priority": 1},
    {"id": "T2 (Superfast)", "arr": 10.25, "dep": 10.45, "pref_plat": 1, "priority": 4},
    {"id": "T3 (Freight)",   "arr": 10.36, "dep": 10.55, "pref_plat": 1, "priority": 1},
]
conflicts_1 = [(0, 1), (1, 2), (2, 3)]
cost_1 = np.array([[3., 12.], [1., 4.], [16., 4.], [4., 1.]])
model_1 = build_compiled_model(4, 2, conflicts_1, cost_1)
qubo_1, _ = model_1.to_qubo(feed_dict={'lambda': 15.0})

sqa_res_1 = oj.SQASampler().sample_qubo(qubo_1, num_reads=300)
_, _, _, best_grid_1 = fast_evaluate(sqa_res_1, 4, 2, conflicts_1, cost_1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

for i, t in enumerate(trains_1):
    p = t["pref_plat"]
    axes[0].barh(p, t["dep"] - t["arr"], left=t["arr"], height=0.35, color=colors[i], edgecolor='black', alpha=0.85)
    axes[0].text((t["arr"] + t["dep"])/2, p, t["id"].split()[0], ha='center', va='center', color='white', fontweight='bold')
axes[0].set_title("Before: Preferred Allocation (Overlapping Collisions!)", fontweight='bold')
axes[0].set_xlabel("Time (Hours)")
axes[0].set_yticks([0, 1])
axes[0].set_yticklabels(["Platform 0", "Platform 1"])
axes[0].grid(axis='x', linestyle='--', alpha=0.5)

for i, t in enumerate(trains_1):
    p = int(np.argmax(best_grid_1[i]))
    axes[1].barh(p, t["dep"] - t["arr"], left=t["arr"], height=0.35, color=colors[i], edgecolor='black', alpha=0.85)
    axes[1].text((t["arr"] + t["dep"])/2, p, t["id"].split()[0], ha='center', va='center', color='white', fontweight='bold')
axes[1].set_title("After: SQA Reassignment (100% Conflict-Free, Cost = 21.0)", fontweight='bold')
axes[1].set_xlabel("Time (Hours)")
axes[1].grid(axis='x', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig("exp1_gantt_comparison.png", dpi=300)
plt.close()

# ==========================================
# EXPERIMENT 2: PENALTY SENSITIVITY (6 Trains, 3 Platforms = 18 Vars)
# ==========================================
print("Running Experiment 2: Penalty Sensitivity Analysis...")
t_2, conf_2, cost_2 = generate_railway_scenario(6, 3, seed=10)
model_2 = build_compiled_model(6, 3, conf_2, cost_2)
penalties = [1, 2, 4, 8, 12, 16, 25, 40]
pen_records = []

for lam_val in penalties:
    qubo_2, _ = model_2.to_qubo(feed_dict={'lambda': float(lam_val)})
    sa_r = oj.SASampler().sample_qubo(qubo_2, num_reads=300)
    sqa_r = oj.SQASampler().sample_qubo(qubo_2, num_reads=300)
    sa_vp, sa_bc, sa_ac, _ = fast_evaluate(sa_r, 6, 3, conf_2, cost_2)
    sqa_vp, sqa_bc, sqa_ac, _ = fast_evaluate(sqa_r, 6, 3, conf_2, cost_2)
    pen_records.append({
        "Penalty": lam_val,
        "SA Valid (%)": round(sa_vp, 2), "SQA Valid (%)": round(sqa_vp, 2),
        "SA Avg Cost": round(sa_ac, 2) if not np.isnan(sa_ac) else None,
        "SQA Avg Cost": round(sqa_ac, 2) if not np.isnan(sqa_ac) else None,
        "SQA Best Cost": sqa_bc if not np.isnan(sqa_bc) else None
    })

pen_df = pd.DataFrame(pen_records)
print("\n--- EXPERIMENT 2: PENALTY SENSITIVITY TABLE ---")
print(pen_df.to_string(index=False))

fig, ax1 = plt.subplots(figsize=(8, 4.5))
ax1.plot(pen_df["Penalty"], pen_df["SQA Valid (%)"], 'o-', label="SQA Valid (%)", color='#1f77b4', linewidth=2)
ax1.plot(pen_df["Penalty"], pen_df["SA Valid (%)"], 's--', label="Classical SA Valid (%)", color='#ff7f0e', linewidth=2)
ax1.set_xlabel(r"Penalty Weight ($\lambda$)")
ax1.set_ylabel("Valid Schedules (%)", color='#1f77b4')
ax1.set_ylim(0, 105)
ax1.grid(True, linestyle='--', alpha=0.5)

ax2 = ax1.twinx()
ax2.plot(pen_df["Penalty"], pen_df["SQA Avg Cost"], '^-', label="SQA Avg Valid Cost", color='#2ca02c', linewidth=2)
ax2.plot(pen_df["Penalty"], pen_df["SA Avg Cost"], 'd:', label="SA Avg Valid Cost", color='#d62728', linewidth=2)
ax2.set_ylabel("Average Valid Assignment Cost", color='#2ca02c')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='lower right')
plt.title("Penalty Sensitivity: Feasibility vs. Assignment Cost (18 Binary Variables)", fontweight='bold')
plt.tight_layout()
plt.savefig("exp2_penalty_sensitivity.png", dpi=300)
plt.close()

# ==========================================
# EXPERIMENT 3: SCALING BENCHMARK (5 Frozen Scenarios)
# ==========================================
print("\nRunning Experiment 3: Scaling Benchmark (8 to 80 Binary Variables)...")
scenarios = [
    (1, 4, 2),   # 8 vars
    (2, 6, 3),   # 18 vars
    (3, 10, 3),  # 30 vars
    (4, 15, 4),  # 60 vars
    (5, 20, 4),  # 80 vars
]
scale_records = []

for scen_id, n_t, n_p in scenarios:
    t_s, conf_s, cost_s = generate_railway_scenario(n_t, n_p, seed=100 + scen_id)
    model_s = build_compiled_model(n_t, n_p, conf_s, cost_s)
    qubo_s, _ = model_s.to_qubo(feed_dict={'lambda': 20.0})

    # Classical SA
    t0 = time.perf_counter()
    sa_r = oj.SASampler().sample_qubo(qubo_s, num_reads=300)
    sa_ms = (time.perf_counter() - t0) * 1000
    sa_vp, sa_bc, sa_ac, _ = fast_evaluate(sa_r, n_t, n_p, conf_s, cost_s)

    # Simulated QA
    t0 = time.perf_counter()
    sqa_r = oj.SQASampler().sample_qubo(qubo_s, num_reads=300)
    sqa_ms = (time.perf_counter() - t0) * 1000
    sqa_vp, sqa_bc, sqa_ac, _ = fast_evaluate(sqa_r, n_t, n_p, conf_s, cost_s)

    scale_records.append({
        "Scenario": scen_id, "Trains": n_t, "Platforms": n_p, "Binary Vars": n_t * n_p,
        "Conflicts": len(conf_s),
        "SA Valid (%)": round(sa_vp, 1), "SQA Valid (%)": round(sqa_vp, 1),
        "SA Best Cost": sa_bc, "SQA Best Cost": sqa_bc,
        "SA Avg Cost": round(sa_ac, 1), "SQA Avg Cost": round(sqa_ac, 1),
        "SA Time (ms)": round(sa_ms, 1), "SQA Time (ms)": round(sqa_ms, 1)
    })

scale_df = pd.DataFrame(scale_records)
print("\n--- EXPERIMENT 3: SCALING BENCHMARK TABLE ---")
print(scale_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].plot(scale_df["Binary Vars"], scale_df["SQA Time (ms)"], 'o-', label="Simulated QA (OpenJij)", linewidth=2)
axes[0].plot(scale_df["Binary Vars"], scale_df["SA Time (ms)"], 's--', label="Classical SA (OpenJij)", linewidth=2)
axes[0].set_xlabel("Number of Binary Variables")
axes[0].set_ylabel("Sampling Runtime (ms) for 300 Reads")
axes[0].set_title("Execution Time Scaling (8 to 80 Variables)", fontweight='bold')
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.5)

axes[1].plot(scale_df["Binary Vars"], scale_df["SQA Valid (%)"], 'o-', label="Simulated QA Valid (%)", linewidth=2)
axes[1].plot(scale_df["Binary Vars"], scale_df["SA Valid (%)"], 's--', label="Classical SA Valid (%)", linewidth=2)
axes[1].set_xlabel("Number of Binary Variables")
axes[1].set_ylabel("Feasible Schedules Found (%)")
axes[1].set_title("Feasibility Retention Across Problem Scale", fontweight='bold')
axes[1].set_ylim(0, 105)
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig("exp3_scaling_benchmark.png", dpi=300)
plt.close()
print("\nSaved updated: exp1_gantt_comparison.png, exp2_penalty_sensitivity.png, exp3_scaling_benchmark.png")

Running Experiment 1: Before vs. After Platform Allocation...
Running Experiment 2: Penalty Sensitivity Analysis...

--- EXPERIMENT 2: PENALTY SENSITIVITY TABLE ---
 Penalty  SA Valid (%)  SQA Valid (%)  SA Avg Cost  SQA Avg Cost  SQA Best Cost
       1          0.00           0.00          NaN           NaN            NaN
       2          0.00           0.00          NaN           NaN            NaN
       4          0.00           0.00          NaN           NaN            NaN
       8          0.00           0.00          NaN           NaN            NaN
      12         46.67          47.33        26.05         25.06           25.0
      16        100.00          96.67        26.66         25.35           25.0
      25        100.00          99.33        28.78         26.08           25.0
      40        100.00          81.00        31.61         26.98           25.0

Running Experiment 3: Scaling Benchmark (8 to 80 Binary Variables)...

--- EXPERIMENT 3: SCALING BENCHMARK TABLE -